In [ ]:
import MetaTrader5 as mt5
import logging
import pandas as pd
import numpy as np
import ta
from sklearn.preprocessing import RobustScaler
from tensorflow.keras.models import load_model
import joblib
import time
from datetime import datetime
import schedule
import sys
import tensorflow as tf
import traceback

# Hàm khởi tạo kết nối MT5
def initialize_mt5():
    if not mt5.initialize():
        logging.error(f"Không thể khởi tạo MetaTrader5, lỗi: {mt5.last_error()}")
        return False
    logging.info("Khởi tạo MetaTrader5 thành công.")
    return True

# Hàm đóng kết nối MT5
def shutdown_mt5():
    mt5.shutdown()
    logging.info("Đóng kết nối MetaTrader5.")

# Hàm kiểm tra kết nối MT5
def is_mt5_connected():
    return mt5.terminal_info() is not None and mt5.account_info() is not None

def reconnect_mt5(max_retries=5, delay=5):
    retries = 0
    while retries < max_retries:
        logging.info(f"Cố gắng kết nối lại MT5 lần {retries + 1}...")
        mt5.shutdown()
        if mt5.initialize():
            logging.info("Kết nối lại MT5 thành công.")
            return True
        logging.warning(f"Lần kết nối thứ {retries + 1} thất bại. Đang chờ {delay} giây...")
        time.sleep(delay)
        retries += 1
    logging.error("Không thể kết nối lại MetaTrader5 sau nhiều lần thử.")
    return False

# Hàm kiểm tra số dư tài khoản
def check_account_balance(required_margin):
    account_info = mt5.account_info()
    if account_info is None:
        logging.error("Không thể lấy thông tin tài khoản.")
        return False
    free_margin = account_info.margin_free
    if free_margin < required_margin:
        logging.warning(f"Không đủ margin. Free Margin: {free_margin}, Required: {required_margin}")
        return False
    return True

# Hàm kiểm tra số lượng lệnh đang mở
def check_open_orders(symbol, max_open_orders=5):
    positions = mt5.positions_get(symbol=symbol)
    if positions is None:
        return 0
    return len(positions)

def check_total_open_orders(max_total_orders=10):
    positions = mt5.positions_get()
    if positions is None:
        return 0
    return len(positions)

# Hàm kiểm tra quy mô lệnh tối đa
def check_max_volume(new_order_volume, max_volume_per_trade=1.0):
    account_info = mt5.account_info()
    if account_info is None:
        logging.error("Không thể lấy thông tin tài khoản.")
        return False
    return new_order_volume <= max_volume_per_trade

# Hàm tính giá trị pip
def calculate_pip(symbol_info):
    if symbol_info is None:
        logging.error("Không có symbol_info để tính pip.")
        return None
    digits = symbol_info.digits
    if digits in [3, 5]:
        pip = 10 ** (-(digits - 1))
    else:
        pip = 10 ** (-digits)
    return pip

# Hàm tính khối lượng giao dịch theo % rủi ro
def calculate_volume(symbol, action, price, sl_price, risk_percentage=0.01):
    account_info = mt5.account_info()
    if account_info is None:
        logging.error("Không thể lấy thông tin tài khoản.")
        return 0.0
    balance = account_info.balance
    
    symbol_info = mt5.symbol_info(symbol)
    if symbol_info is None:
        logging.error(f"Không thể lấy thông tin symbol {symbol}.")
        return 0.0
    
    pip = calculate_pip(symbol_info)
    stop_loss_pips = abs(price - sl_price) / pip
    
    base_currency = symbol[:3]
    quote_currency = symbol[3:]
    account_currency = account_info.currency
    
    pip_value_base = symbol_info.trade_contract_size * pip
    
    if quote_currency == account_currency:
        pip_value = pip_value_base
    else:
        conversion_pair = quote_currency + account_currency
        if mt5.symbol_info(conversion_pair) is not None:
            tick = mt5.symbol_info_tick(conversion_pair)
            if tick is None:
                logging.error(f"Không thể lấy tick cho {conversion_pair}.")
                return 0.0
            conversion_rate = tick.bid
            pip_value = pip_value_base * conversion_rate
        else:
            conversion_pair = account_currency + quote_currency
            if mt5.symbol_info(conversion_pair) is not None:
                tick = mt5.symbol_info_tick(conversion_pair)
                if tick is None:
                    logging.error(f"Không thể lấy tick cho {conversion_pair}.")
                    return 0.0
                conversion_rate = tick.bid
                pip_value = pip_value_base / conversion_rate
            else:
                logging.error(f"Không tìm thấy cặp chuyển đổi cho {quote_currency} sang {account_currency}.")
                return 0.0
    
    risk_amount = balance * risk_percentage
    volume = risk_amount / (stop_loss_pips * pip_value)
    volume = min(volume, symbol_info.volume_max)
    volume = max(volume, symbol_info.volume_min)
    return round(volume, 2)

# Hàm thực hiện giao dịch
def execute_trade(symbol, action, predicted_price, max_open_orders=2, max_total_orders=10, max_volume_per_trade=1.0, risk_percentage=0.01):
    symbol_info = mt5.symbol_info(symbol)
    if symbol_info is None:
        logging.error(f"Symbol {symbol} không tồn tại.")
        return
    if not symbol_info.visible:
        logging.info(f"Symbol {symbol} không hiển thị. Cố gắng thêm vào Market Watch.")
        if not mt5.symbol_select(symbol, True):
            logging.error(f"Không thể thêm {symbol} vào Market Watch.")
            return

    total_open_orders = check_open_orders(symbol, max_open_orders)
    if total_open_orders >= max_open_orders:
        logging.info(f"Đã đạt số lượng lệnh mở tối đa cho {symbol} ({max_open_orders}).")
        return

    if check_total_open_orders(max_total_orders) >= max_total_orders:
        logging.info(f"Đã đạt tổng số lệnh mở tối đa ({max_total_orders}).")
        return

    pip = calculate_pip(symbol_info)
    min_pip_diff = 10 * pip
    price = mt5.symbol_info_tick(symbol).ask if action == "BUY" else mt5.symbol_info_tick(symbol).bid
    deviation = 10
    stops_level = symbol_info.trade_stops_level * pip

# Logic SL/TP mới theo RR 1:1
    if action.upper() == "BUY":
        # Vào khi predicted < price
        if predicted_price >= price:
            logging.warning(f"Predicted price ({predicted_price}) không nhỏ hơn giá hiện tại ({price}) cho BUY.")
            return
        # SL = predicted_price
        sl_price = predicted_price
        # Khoảng cách entry->SL
        distance = price - sl_price
        # TP đối xứng
        tp_price = price + distance
        order_type = mt5.ORDER_TYPE_BUY

    elif action.upper() == "SELL":
        # Vào khi predicted > price
        if predicted_price <= price:
            logging.warning(f"Predicted price ({predicted_price}) không lớn hơn giá hiện tại ({price}) cho SELL.")
            return
        # SL = predicted_price
        sl_price = predicted_price
        # Khoảng cách entry->SL
        distance = sl_price - price
        # TP đối xứng
        tp_price = price - distance
        order_type = mt5.ORDER_TYPE_SELL

    else:
        logging.error(f"Hành động không hợp lệ: {action}")
        return

    if action.upper() == "BUY":
        if sl_price >= price or tp_price <= price:
            logging.error(f"SL {sl_price} hoặc TP {tp_price} không hợp lệ cho lệnh BUY (yêu cầu SL < {price} < TP)")
            return
    elif action.upper() == "SELL":
        if sl_price <= price or tp_price >= price:
            logging.error(f"SL {sl_price} hoặc TP {tp_price} không hợp lệ cho lệnh SELL (yêu cầu TP < {price} < SL)")
            return

    if abs(price - sl_price) < stops_level or abs(price - tp_price) < stops_level:
        logging.warning(f"SL/TP không hợp lệ cho {symbol} theo quy định broker.")
        return

    from decimal import Decimal, ROUND_HALF_UP

    def round_to_digits(value, digits):
        return float(Decimal(str(value)).quantize(Decimal(f'0.{"0" * digits}'), rounding=ROUND_HALF_UP))

    price = round_to_digits(price, symbol_info.digits)
    sl_price = round_to_digits(sl_price, symbol_info.digits)
    tp_price = round_to_digits(tp_price, symbol_info.digits)

    logging.info(f"Symbol: {symbol} | Action: {action} | Price: {price} | SL: {sl_price} | TP: {tp_price}")

    open_buy_orders = 0
    open_sell_orders = 0
    positions = mt5.positions_get(symbol=symbol)
    if positions is not None:
        for pos in positions:
            if pos.type == mt5.POSITION_TYPE_BUY:
                open_buy_orders += 1
            elif pos.type == mt5.POSITION_TYPE_SELL:
                open_sell_orders += 1

    if action == "BUY" and open_buy_orders >= 1:
        logging.info(f"Đã đạt số lượng lệnh BUY tối đa cho {symbol} (1).")
        return
    if action == "SELL" and open_sell_orders >= 1:
        logging.info(f"Đã đạt số lượng lệnh SELL tối đa cho {symbol} (1).")
        return

    if check_open_orders(symbol, max_open_orders) >= max_open_orders:
        logging.info(f"Đã đạt số lượng lệnh mở tối đa cho {symbol} ({max_open_orders}).")
        return
    if check_total_open_orders(max_total_orders) >= max_total_orders:
        logging.info(f"Đã đạt tổng số lệnh mở tối đa ({max_total_orders}).")
        return

    volume = calculate_volume(symbol, action, price, sl_price, risk_percentage)
    if volume == 0.0:
        logging.warning("Không thể tính toán volume hợp lệ.")
        return

    volume = max(symbol_info.volume_min, volume)
    if symbol_info.volume_max > 0:
        volume = min(symbol_info.volume_max, volume)
    if symbol_info.volume_step > 0:
        steps = round((volume - symbol_info.volume_min) / symbol_info.volume_step)
        volume = symbol_info.volume_min + steps * symbol_info.volume_step
    logging.info(f"Adjusted volume for {symbol}: {volume}")

    if not check_max_volume(volume, max_volume_per_trade):
        logging.warning(f"Kích thước lệnh {volume} vượt quá quy mô tối đa cho phép ({max_volume_per_trade}).")
        return

    leverage = mt5.account_info().leverage
    required_margin = (volume * price) / leverage
    if not check_account_balance(required_margin):
        logging.warning(f"Không đủ margin để mở lệnh {action} cho {symbol}.")
        return

    request = {
        "action": mt5.TRADE_ACTION_DEAL,
        "symbol": symbol,
        "volume": volume,
        "type": order_type,
        "price": price,
        "sl": round(sl_price, symbol_info.digits),
        "tp": round(tp_price, symbol_info.digits),
        "deviation": deviation,
        "magic": 881111,
        "comment": "15NU00",
        "type_time": mt5.ORDER_TIME_GTC,
        "type_filling": mt5.ORDER_FILLING_IOC,
    }

    result = mt5.order_send(request)
    if result is None:
        error = mt5.last_error()
        logging.error(f"Lỗi khi gửi lệnh: {error[0]} - {error[1]}")
        return
    if result.retcode != mt5.TRADE_RETCODE_DONE:
        logging.error(f"Lỗi khi gửi lệnh: {result.retcode}")
    else:
        logging.info(f"Giao dịch thành công: {action} {symbol} tại giá {price} | SL: {sl_price} | TP: {tp_price} | Volume: {volume}")

# Hàm lấy dữ liệu lịch sử
def get_historical_data(symbol, timeframe, look_back, buffer=500, max_attempts=5, delay=1):
    total_look_back = look_back + buffer
    logging.info(f"Lấy tổng cộng {total_look_back} phiên dữ liệu cho {symbol} (timeframe={timeframe})")
    
    last_valid_data = None
    
    for attempt in range(1, max_attempts + 1):
        logging.info(f"[{symbol}] Lần thử lấy dữ liệu thứ {attempt}/{max_attempts}...")
        
        if not mt5.symbol_select(symbol, True):
            logging.error(f"Không thể chọn symbol {symbol} trong Market Watch.")
            time.sleep(delay)
            continue
        
        rates = mt5.copy_rates_from_pos(symbol, timeframe, 1, total_look_back)
        
        if rates is None or len(rates) < total_look_back:
            logging.warning(f"[{symbol}] Lần thử {attempt}: Không đủ dữ liệu, nhận được {len(rates) if rates else 0} thanh nến.")
            time.sleep(delay)
            continue
        
        df = pd.DataFrame(rates)
        df['Time'] = pd.to_datetime(df['time'], unit='s')
        df.set_index('Time', inplace=True)
        df.drop('time', axis=1, inplace=True)
        df.rename(columns={'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'tick_volume': 'Volume'}, inplace=True)
        
        if df[features_list].isna().any().any():
            logging.warning(f"[{symbol}] Lần thử {attempt}: Dữ liệu chứa NaN.")
            time.sleep(delay)
            continue
        
        last_valid_data = df
        logging.info(f"[{symbol}] Lần thử {attempt}: Lấy dữ liệu thành công, số lượng thanh nến: {len(df)}")
        time.sleep(delay)
    
    if last_valid_data is None:
        logging.error(f"[{symbol}] Không thể lấy dữ liệu hợp lệ sau {max_attempts} lần thử.")
        return None
    
    logging.info(f"Số lượng dữ liệu sau khi xử lý cho {symbol}: {len(last_valid_data)}")
    print(f"5 dòng cuối của dữ liệu cho {symbol} (timeframe={timeframe}):")
    print(last_valid_data.tail(5))
    
    return last_valid_data

# Danh sách đặc trưng
features_list = ['Open', 'High', 'Low', 'Close']

# Hàm tạo tín hiệu giao dịch
def generate_signal(predicted_price, current_price, min_pip_diff):
    price_diff = predicted_price - current_price
    if price_diff >= min_pip_diff:
        return "SELL"
    elif price_diff <= -min_pip_diff:
        return "BUY"
    return "HOLD"

def run_trading_cycle(pairs, time_steps_m5, time_steps_m15, time_steps_m30, time_steps_h1, features_list, scaler_m5, scaler_m15, scaler_m30, scaler_h1, scaler_target, model):
    """Hàm chứa logic chính để lấy dữ liệu, dự đoán và giao dịch cho một chu kỳ."""
    logging.info("=" * 20 + " BẮT ĐẦU CHU KỲ XỬ LÝ " + "=" * 20)
    start_time = time.time()

    try:
        if not is_mt5_connected():
            logging.warning("Kết nối MT5 bị gián đoạn. Cố gắng kết nối lại...")
            if not reconnect_mt5():
                logging.error("Không thể kết nối lại MT5. Bỏ qua chu kỳ này.")
                logging.info("--- Kết thúc chu kỳ xử lý (Lỗi kết nối MT5) ---")
                return

        for pair in pairs:
            logging.info(f"[{pair}] Bắt đầu xử lý...")
            pair_start_time = time.time()

            logging.info(f"[{pair}] Đang lấy dữ liệu M5, M15, M30, H1...")
            df_m5 = get_historical_data(pair, mt5.TIMEFRAME_M5, time_steps_m5, buffer=0, max_attempts=5, delay=1)
            df_m15 = get_historical_data(pair, mt5.TIMEFRAME_M15, time_steps_m15, buffer=0, max_attempts=5, delay=1)
            df_m30 = get_historical_data(pair, mt5.TIMEFRAME_M30, time_steps_m30, buffer=0, max_attempts=5, delay=1)
            df_h1 = get_historical_data(pair, mt5.TIMEFRAME_H1, time_steps_h1, buffer=0, max_attempts=5, delay=1)

            if df_m5 is None or df_m15 is None or df_m30 is None or df_h1 is None:
                logging.warning(f"[{pair}] Không đủ dữ liệu. Bỏ qua.")
                continue

            if df_m5[features_list].isna().any().any() or df_m15[features_list].isna().any().any() or df_m30[features_list].isna().any().any() or df_h1[features_list].isna().any().any():
                logging.warning(f"[{pair}] Dữ liệu chứa NaN. Bỏ qua.")
                continue

            symbol_info = mt5.symbol_info(pair)
            if symbol_info is None:
                logging.warning(f"[{pair}] Không thể lấy thông tin symbol. Bỏ qua.")
                continue
            price_digits = symbol_info.digits

            logging.debug(f"[{pair}] Chuẩn bị dữ liệu cho model...")
            scaled_m5 = scaler_m5.transform(df_m5[features_list])
            scaled_m15 = scaler_m15.transform(df_m15[features_list])
            scaled_m30 = scaler_m30.transform(df_m30[features_list])
            scaled_h1 = scaler_h1.transform(df_h1[features_list])

            # Không đảo ngược thứ tự dữ liệu, giữ nguyên từ cũ đến mới
            scaled_m5 = scaled_m5[-time_steps_m5:]
            scaled_m15 = scaled_m15[-time_steps_m15:]
            scaled_m30 = scaled_m30[-time_steps_m30:]
            scaled_h1 = scaled_h1[-time_steps_h1:]

            X_m5 = scaled_m5.reshape(1, time_steps_m5, len(features_list))
            X_m15 = scaled_m15.reshape(1, time_steps_m15, len(features_list))
            X_m30 = scaled_m30.reshape(1, time_steps_m30, len(features_list))
            X_h1 = scaled_h1.reshape(1, time_steps_h1, len(features_list))

            logging.info(f"[{pair}] Thực hiện dự đoán...")
            predicted_scaled = model.predict([X_m5, X_m15, X_m30, X_h1], verbose=0)
            predicted_price = scaler_target.inverse_transform(predicted_scaled)[0][0]
            current_price = df_h1['Close'].iloc[-1]

            pip = calculate_pip(symbol_info)
            min_pip_diff = 10 * pip
            signal = generate_signal(predicted_price, current_price, min_pip_diff)

            logging.info(f"[{pair}] Giá H1 gần nhất: {current_price:.{price_digits}f} | "
                         f"Giá dự đoán: {predicted_price:.{price_digits}f} | "
                         f"Chênh lệch (pips): {(predicted_price - current_price)/pip:.1f} | "
                         f"Tín hiệu: {signal}")

            if signal in ["BUY", "SELL"]:
                logging.info(f"[{pair}] Có tín hiệu {signal}. Thực hiện giao dịch...")
                execute_trade(
                    symbol=pair,
                    action=signal,
                    predicted_price=predicted_price,
                    max_open_orders=1,              # Số lệnh mở cho mỗi cặp
                    max_total_orders=3,             # Số lệnh mở tổng cộng
                    max_volume_per_trade=1.0,
                    risk_percentage=0.08
                )
            else:
                logging.info(f"[{pair}] Không có tín hiệu giao dịch.")
 
            pair_end_time = time.time()
            logging.info(f"[{pair}] Hoàn thành xử lý trong {pair_end_time - pair_start_time:.2f} giây.")

    except Exception as e:
        logging.error(f"!!! Lỗi nghiêm trọng trong chu kỳ xử lý: {e}")
        logging.error(traceback.format_exc())

    finally:
        end_time = time.time()
        logging.info(f"--- Kết thúc chu kỳ xử lý trong {end_time - start_time:.2f} giây ---")

# Hàm chính
def main():
    log_filename = 'bot15NU00.log'
    log_format = '%(asctime)s - %(levelname)s - [%(funcName)s:%(lineno)d] - %(message)s'
    log_datefmt = '%Y-%m-%d %H:%M:%S'

    logging.basicConfig(
        filename=log_filename,
        filemode='a',
        level=logging.INFO,
        format=log_format,
        datefmt=log_datefmt,
        encoding='utf-8'
    )

    logger = logging.getLogger()
    if not any(isinstance(handler, logging.StreamHandler) for handler in logger.handlers):
        console_handler = logging.StreamHandler(sys.stdout)
        console_handler.setLevel(logging.INFO)
        formatter = logging.Formatter(log_format, datefmt=log_datefmt)
        console_handler.setFormatter(formatter)
        logger.addHandler(console_handler)

    logging.info("="*10 + " CHƯƠNG TRÌNH GIAO DỊCH KHỞI ĐỘNG " + "="*10)

    if not initialize_mt5():
        logging.critical("Kết nối MT5 ban đầu thất bại. Thoát chương trình.")
        return

    try:
        logging.info("Đang tải scaler và model...")
        scaler_m5 = joblib.load('scaler_m5_extended.save')
        scaler_m15 = joblib.load('scaler_m15_extended.save')
        scaler_m30 = joblib.load('scaler_m30_extended.save')
        scaler_h1 = joblib.load('scaler_h1_extended.save')
        scaler_target = joblib.load('scaler_target.save')
        model = load_model('NU_4k_2n.keras')
        logging.info("Tải scaler và mô hình thành công.")

        pairs = ["NZDUSD"]
        time_steps_m5 = 800
        time_steps_m15 = 700
        time_steps_m30 = 600
        time_steps_h1 = 600
        features_list = ['Open', 'High', 'Low', 'Close']

        logging.info("Lên lịch chạy Job mỗi giờ vào phút :01...")
        schedule.every().hour.at(":01").do(
            run_trading_cycle,
            pairs=pairs,
            time_steps_m5=time_steps_m5,
            time_steps_m15=time_steps_m15,
            time_steps_m30=time_steps_m30,
            time_steps_h1=time_steps_h1,
            features_list=features_list,
            scaler_m5=scaler_m5,
            scaler_m15=scaler_m15,
            scaler_m30=scaler_m30,
            scaler_h1=scaler_h1,
            scaler_target=scaler_target,
            model=model
        )

        logging.info("Bắt đầu vòng lặp chờ lịch chạy...")
        while True:
            schedule.run_pending()
            time.sleep(1)

    except KeyboardInterrupt:
        logging.info("Script bị dừng bởi người dùng (Ctrl+C).")
    except Exception as e:
        logging.critical(f"Lỗi không xác định ở vòng lặp chính: {e}")
        logging.critical(traceback.format_exc())
    finally:
        logging.info("Đang đóng kết nối MetaTrader5...")
        shutdown_mt5()
        logging.info("="*10 + " CHƯƠNG TRÌNH KẾT THÚC " + "="*10)

if __name__ == "__main__":
    main()